# 03 · Preprocesado común: oraciones, tokens y raíces

Tercer notebook del preprocesado (E2). Parte de los cuatro ficheros `*_limpio.jsonl` de los notebooks 01 y 02 y les aplica a todos **la misma función de preprocesado**:

1. Normalización Unicode (NFC).
2. División en oraciones.
3. Tokenización (`tokens`, para embeddings).
4. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF).

El resultado son los `*_procesado.jsonl`, con el formato de `ESQUEMA_DATOS.md`.

## 0. Preparación

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from nltk.tokenize import sent_tokenize

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_DATOS = Path("../data/processed")
FICHEROS = ["fallacyes_proto", "fallacyes_espont", "fallacyes_political", "congreso"]

corpus = {
    nombre: pd.read_json(CARPETA_DATOS / f"{nombre}_limpio.jsonl", lines=True, dtype=False)
    for nombre in FICHEROS
}
for nombre, tabla in corpus.items():
    print(f"{nombre}: {len(tabla)} textos")

fallacyes_proto: 3319 textos
fallacyes_espont: 1840 textos
fallacyes_political: 1951 textos
congreso: 2468 textos


## 1. Normalización Unicode (NFC)

Una letra con tilde se puede guardar de dos formas: como un solo carácter (`é`) o como la letra más una tilde suelta (`e` + `´`). Se ven igual, pero para Python son cadenas distintas, y el Word2Vec las trataría como dos palabras. NFC lo deja todo en la primera forma. **No quita las tildes.**

In [2]:
compuesta = "país"  # "i" + tilde suelta
print(compuesta, "==", "país", "→", compuesta == "país")
print("con NFC →", unicodedata.normalize("NFC", compuesta) == "país")

for nombre, tabla in corpus.items():
    cambian = tabla["texto"].map(lambda texto: unicodedata.normalize("NFC", texto) != texto)
    print(f"{nombre}: textos que cambian con NFC: {cambian.sum()}")

país == país → False
con NFC → True
fallacyes_proto: textos que cambian con NFC: 0
fallacyes_espont: textos que cambian con NFC: 0
fallacyes_political: textos que cambian con NFC: 0
congreso: textos que cambian con NFC: 0


En nuestros datos no cambia ningún texto. Se aplica igualmente: es una red de seguridad para textos nuevos (copiados de un PDF o escritos en un Mac) y es una línea de código.

## 2. División en oraciones: qué falla con `sent_tokenize`

`sent_tokenize` de NLTK usa el modelo Punkt para español. Punkt no corta en todos los puntos: tiene una lista de abreviaturas tras las que no corta. Se mira dónde se equivoca en nuestros datos.

In [3]:
pruebas = [
    "La Sra. García dijo que no. El Sr. López tampoco.",
    "Lo dijo el Prof. Martínez en clase.",
    "Vive en St. Louis desde hace años.",
    "Lo aprobó EE. UU. y luego Europa.",
    "Según el art. 155 de la Constitución.",
    "Vinieron todos: Juan, Ana, etc. Pero nadie habló.",
]
for prueba in pruebas:
    print(sent_tokenize(prueba, language="spanish"))

['La Sra.', 'García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof.', 'Martínez en clase.']
['Vive en St.', 'Louis desde hace años.']
['Lo aprobó EE.', 'UU. y luego Europa.']
['Según el art.', '155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']


In [4]:
# Cuántas veces sale cada abreviatura en cada corpus
ABREVIATURAS_A_BUSCAR = {
    "Sra.": r"\bSra\.", "Prof.": r"\bProf\.", "Dr. / Dra.": r"\bDra?\.", "St.": r"\bSt\.",
    "art.": r"\barts?\.", "núm.": r"\bnúm\.", "cap.": r"\bcap\.", "vol.": r"\bvol\.",
    "EE. UU.": r"EE\. ?UU\.", "etc.": r"\betc\.",
}
pd.DataFrame({
    nombre: {abreviatura: tabla["texto"].str.count(patron).sum() for abreviatura, patron in ABREVIATURAS_A_BUSCAR.items()}
    for nombre, tabla in corpus.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
Sra.,24,0,0,0
Prof.,8,0,0,0
Dr. / Dra.,12,0,0,0
St.,6,0,0,0
art.,0,0,1,0
núm.,0,0,0,1
cap.,0,0,0,4
vol.,0,0,0,3
EE. UU.,1,2,2,0
etc.,2,18,3,6


### 2.1 Oraciones muy largas

El texto de Political viene de una transcripción automática (WhisperX), que a veces puntúa poco. Se mira cuánto miden las oraciones de cada corpus.

In [5]:
oraciones_de_prueba = {
    nombre: tabla["texto"].map(lambda texto: sent_tokenize(texto, language="spanish")).explode()
    for nombre, tabla in corpus.items()
}
pd.DataFrame({
    nombre: {
        "oraciones": len(oraciones),
        "palabras (mediana)": oraciones.str.split().str.len().median(),
        "palabras (percentil 95)": oraciones.str.split().str.len().quantile(0.95),
        "palabras (máximo)": oraciones.str.split().str.len().max(),
        "más de 80 palabras": (oraciones.str.split().str.len() > 80).sum(),
    }
    for nombre, oraciones in oraciones_de_prueba.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones,5859.0,2745.0,4290.00,68449.0
palabras (mediana),13.0,16.0,16.00,20.0
palabras (percentil 95),30.0,38.0,44.55,61.0
palabras (máximo),90.0,83.0,191.00,252.0
más de 80 palabras,1.0,1.0,11.00,1125.0


In [6]:
# Ejemplos de oraciones de más de 80 palabras
for nombre in ["fallacyes_political", "congreso"]:
    oraciones = oraciones_de_prueba[nombre]
    largas = oraciones[oraciones.str.split().str.len() > 80]
    print(f"--- {nombre}")
    for oracion in largas.sample(2, random_state=1):
        print("   ", oracion[:350], "...\n")

--- fallacyes_political
    Que han bajado los porcentajes de gente que abandonaba pronto los estudios, los porcentajes de fracaso escolar, que le hemos dado un impulso muy importante a la formación profesional, que tenemos que adecuar lo que se estudie a lo que sea la realidad del mercado laboral, que vamos a crear 100.000 plazas, como le he señalado antes, en materia de for ...

    Yo creo que estas elecciones son decisivas porque si gobierna la derecha todo el mundo sabe que vamos a ir a peor y que se van a cuestionar derechos que hasta ahora estaban asentados y eso va a ocurrir si gobiernan PP, Ciudadanos y Vox y también si gobierna Ciudadanos con el Partido Socialista porque eso también es un gobierno de derechas y yo creo ...

--- congreso
    Aunque en nuestro país la Generalitat de Catalunya ya garantizara oficinas móviles para estos municipios, es a la banca a quien se tiene que imponer la obligatoriedad de garantizar este servicio y la gratuidad de hacer la retirada de efect

## 3. División en oraciones

**Decisión:** se usa Punkt (el modelo de `sent_tokenize`) añadiéndole las abreviaturas que le faltan, **solo las que salen en nuestros datos** (sección 2): *Sra., Dra., Prof., St., EE., art., núm., cap., vol.* Punkt guarda sus abreviaturas en minúscula y sin el punto final. Añadirlas es la forma habitual de adaptar Punkt a un dominio.

- *EE.* basta para *EE. UU.*: *UU.* ya estaba en la lista.
- *etc.* seguido de mayúscula se deja cortar: casi siempre la frase termina ahí (*"Juan, Ana, etc. Pero nadie habló."*).
- **Las oraciones largas no se parten** (sección 2.1): son frases largas de verdad, no errores. No molestan al BoW ni al Word2Vec, y la más larga (252 palabras) cabe en un transformer (unas 350 palabras como máximo). Lo que no cabe son las **intervenciones enteras** del Congreso (mediana de 394 palabras); con qué unidad se pasan por el detector se decidirá al usarlo. La jerarquía intervención → oración permite cualquier opción.

Se crea un tokenizador propio en vez de modificar el que usa `sent_tokenize` por dentro, para que el cambio quede a la vista.

In [7]:
from nltk.tokenize import PunktTokenizer

ABREVIATURAS_QUE_FALTAN = {"sra", "dra", "prof", "st", "ee", "art", "núm", "cap", "vol"}

tokenizador_de_oraciones = PunktTokenizer("spanish")
# _params es un atributo interno de NLTK, pero es la forma habitual de añadir abreviaturas a Punkt
tokenizador_de_oraciones._params.abbrev_types.update(ABREVIATURAS_QUE_FALTAN)


def dividir_en_oraciones(texto):
    return tokenizador_de_oraciones.tokenize(texto)


# Los casos de la sección 2, más alguno en el que sí hay que cortar
for prueba in pruebas + ["Lo aprobó EE. UU. Luego Europa.", "Lo dice el núm. 3. Y el cap. 2 también."]:
    print(dividir_en_oraciones(prueba))

['La Sra. García dijo que no.', 'El Sr. López tampoco.']
['Lo dijo el Prof. Martínez en clase.']
['Vive en St. Louis desde hace años.']
['Lo aprobó EE. UU. y luego Europa.']
['Según el art. 155 de la Constitución.']
['Vinieron todos: Juan, Ana, etc.', 'Pero nadie habló.']
['Lo aprobó EE. UU.', 'Luego Europa.']
['Lo dice el núm. 3.', 'Y el cap. 2 también.']


In [8]:
# Cuántas oraciones salen antes y después del cambio
oraciones_nuevas = {nombre: tabla["texto"].map(dividir_en_oraciones) for nombre, tabla in corpus.items()}
pd.DataFrame({
    nombre: {
        "oraciones antes": len(oraciones_de_prueba[nombre]),
        "oraciones ahora": oraciones_nuevas[nombre].str.len().sum(),
    }
    for nombre in corpus
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
oraciones antes,5859,2745,4290,68449
oraciones ahora,5819,2745,4289,68441


In [9]:
# Ejemplos de las prototípicas en los que ha cambiado la división
proto = corpus["fallacyes_proto"]["texto"]
antes = proto.map(lambda texto: sent_tokenize(texto, language="spanish"))
ahora = oraciones_nuevas["fallacyes_proto"]
for texto_antes, texto_ahora in list(zip(antes[antes != ahora], ahora[antes != ahora]))[:4]:
    print("antes:", texto_antes)
    print("ahora:", texto_ahora, "\n")

antes: ['La Sra.', 'Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.']
ahora: ['La Sra. Miller dio clases a tu hermano mayor, que era el payaso de la clase.', 'Decide ser estricta contigo desde el primer día de clase porque cree que tú también serás un payaso.'] 

antes: ['La Sra.', 'Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.']
ahora: ['La Sra. Miller dará clases a ti y puede tener sus propias percepciones y expectativas sobre tu comportamiento en clase.', 'Esto no necesariamente significa que te tratará de manera diferente porque tu hermano mayor fue un payaso en su clase.'] 

antes: ['Todos los profesores de escritura deben ser increíbles porque la Sra.', 'López lo es.']
aho

## 4. Tokenización: qué tokenizador usar

Se comparan los tres tokenizadores de NLTK vistos en clase:
- **Treebank** (`TreebankWordTokenizer`): reglas pensadas para el inglés del Penn Treebank.
- **wordpunct** (`wordpunct_tokenize`): separa cualquier secuencia de letras/números de cualquier secuencia de signos.
- **casual** (`casual_tokenize`): pensado para texto de redes sociales (emoticonos, menciones, palabras alargadas).

Primero se mira qué casos difíciles hay en nuestros datos y cuántas veces salen.

In [10]:
CASOS_DIFICILES = {
    "porcentaje (72%, 0,5 %)": r"\d\s?%",
    "decimal con coma (1,5)": r"\b\d+,\d+\b",
    "miles con punto (10.000)": r"\b\d{1,3}(?:\.\d{3})+\b",
    "palabra con guion (Pain-Away)": r"\b\w+-\w+\b",
    "apóstrofo (McDonald's, l’article)": r"\b\w+['’]\w+\b",
    "comillas «»": r"«",
    "comillas \"\"": r"\"",
    "signos de apertura ¿ ¡": r"[¿¡]",
    "puntos suspensivos": r"\.\.\.|…",
    "sigla con puntos (P.N.G., EE. UU.)": r"\b(?:[A-Z]{1,2}\.\s?){2,}",
    "emoticono (xD, :) )": r"(?<!\w)(?:[:;=]-?[)(DPp]|xD|XD)(?!\w)",
}
pd.DataFrame({
    nombre: {caso: tabla["texto"].str.count(patron).sum() for caso, patron in CASOS_DIFICILES.items()}
    for nombre, tabla in corpus.items()
})

,fallacyes_proto,fallacyes_espont,fallacyes_political,congreso
"porcentaje (72%, 0,5 %)",37,31,117,2031
"decimal con coma (1,5)",4,7,35,649
miles con punto (10.000),34,8,145,0
palabra con guion (Pain-Away),62,36,19,1256
"apóstrofo (McDonald's, l’article)",11,2,0,6542
comillas «»,0,3,0,640
"comillas """"",753,271,2,0
signos de apertura ¿ ¡,645,153,452,6294
puntos suspensivos,56,119,49,805
"sigla con puntos (P.N.G., EE. UU.)",13,2,2,5


In [11]:
from nltk.tokenize import TreebankWordTokenizer, casual_tokenize, wordpunct_tokenize

TOKENIZADORES = {
    "treebank": TreebankWordTokenizer().tokenize,
    "wordpunct": wordpunct_tokenize,
    "casual": casual_tokenize,
}

# Frases de nuestros corpus (algunas recortadas); las tres últimas son inventadas para probar siglas y emoticonos
frases_dificiles = [
    "¿Sabía que su tío fue encarcelado por malversar 1,5 millones de dólares?",       # proto
    "Preguntamos a más de 10.000 científicos y el 72% dijo que sí.",                  # proto
    "Pain-Away funcionó para mí, así que seguro que funciona para ti.",               # proto
    "Hamburguesas McDonald's, más de 99.000 millones servidas.",                      # proto
    "Es que no has atinado ni una sobre mi. xD Todo el mundo dice que soy muy majo",  # espont
    "Según la RAE, postula el «principio de igualdad de derechos».",                  # espont
    "¿Indefendible?, jajjajaaaaaaaaa.",                                               # espont
    "Derogar l’article…, l’apartat b), perdó, de l’article 6.2.",                     # congreso
    "Pero cuando miramos el crecimiento del P.N.G. este año...",                      # proto
    "El paro ha bajado un 0,5 % en EE. UU. desde 2023.",
    "El COVID-19 y la DANA del 29-O.",
    "Muy bien explicado :)",
]
for frase in frases_dificiles:
    print(frase)
    for nombre, tokenizar in TOKENIZADORES.items():
        print(f"   {nombre:<9}", tokenizar(frase))
    print()

¿Sabía que su tío fue encarcelado por malversar 1,5 millones de dólares?
   treebank  ['¿Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1,5', 'millones', 'de', 'dólares', '?']
   wordpunct ['¿', 'Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1', ',', '5', 'millones', 'de', 'dólares', '?']
   casual    ['¿', 'Sabía', 'que', 'su', 'tío', 'fue', 'encarcelado', 'por', 'malversar', '1,5', 'millones', 'de', 'dólares', '?']

Preguntamos a más de 10.000 científicos y el 72% dijo que sí.
   treebank  ['Preguntamos', 'a', 'más', 'de', '10.000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']
   wordpunct ['Preguntamos', 'a', 'más', 'de', '10', '.', '000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']
   casual    ['Preguntamos', 'a', 'más', 'de', '10.000', 'científicos', 'y', 'el', '72', '%', 'dijo', 'que', 'sí', '.']

Pain-Away funcionó para mí, así que seguro que funciona para ti.
   treebank  ['Pain-Away', 'funci

### 4.1 Resultado: se usa `casual_tokenize`

| | Treebank | wordpunct | casual |
|---|---|---|---|
| **¿ ¡ « »** | ❌ los deja pegados: `¿Sabía`, `«principio` | ✅ | ✅ |
| **Comillas ""** | ❌ las cambia por `` `` `` y `''` (modifica el texto) | ❌ las junta con otros signos: `".` | ✅ |
| **Cifras** (1,5 · 10.000 · 6.2) | ✅ | ❌ las rompe: `1` `,` `5` | ✅ |
| **Palabras con guion** (Pain-Away) | ✅ | ❌ | ✅ |
| **Apóstrofo recto** (McDonald's) | ❌ separa el `'s` del inglés | ❌ | ✅ |
| **Signos seguidos** (`?,` `»,` `),`) | ✅ | ❌ los junta en un token | ✅ |
| **Emoticonos** (`:)`, `xD`) | ❌ rompe `:)` | ✅ | ✅ |
| **Siglas con puntos** (P.N.G., EE. UU.) | ✅ | ❌ | ❌ las rompe: `P` `.` `N` `.` |
| **Apóstrofo curvo** (l’article, catalán) | ✅ | ❌ | ❌ `l` `’` `article` |
| **Sigla con número** (COVID-19) | ✅ | ❌ | ❌ `COVID` `-` `19` |

- **wordpunct** es el peor: rompe cifras, palabras compuestas y siglas, y pega signos entre sí. En el Congreso hay miles de cifras y porcentajes.
- **Treebank** está hecho para el inglés: no separa los signos de apertura del español (que salen en todos los corpus) y reescribe las comillas.
- **casual** acierta en casi todo. Falla en tres casos, que se aceptan porque son raros o no importan:
  - siglas con puntos: unas 20 en todos los corpus;
  - *COVID-19* y parecidos: pocos;
  - apóstrofo curvo: casi todo es catalán (*l’article*), que no es el idioma de los modelos. Al partirlo, al menos *article* queda como palabra suelta.

**Opciones de `casual_tokenize`:** se dejan las de por defecto. Conserva las mayúsculas (`preserve_case=True`), porque los `tokens` son para embeddings y tienen que coincidir con el vocabulario de los preentrenados. No acorta las letras repetidas (`reduce_len=False`): *jajjajaaaaaaaaa* sale solo en un puñado de textos, y los `tokens` se guardan sin modificar.

## 5. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF)

`raices_bow` es la versión de cada oración para BoW y TF-IDF:
- en minúsculas;
- sin stopwords;
- sin puntuación (salvo `?` y `!`, ver 5.1);
- con las cifras cambiadas por la marca `<num>` (ver 5.2);
- y con cada palabra reducida a su raíz con el stemmer Snowball de NLTK.

Así *país*, *pais* y *países* cuentan como la misma palabra, y el vocabulario es más pequeño.

**Solo para BoW / TF-IDF.** Los `tokens` (para embeddings) lo conservan todo: los preentrenados tienen vectores para las stopwords, el Word2Vec las necesita como contexto y los transformers usan el texto tal cual.

### 5.1 Qué stopwords se conservan

La lista de NLTK en español (313 palabras) quita palabras que pueden delatar una falacia. Para elegir cuáles se conservan, se mide en qué porcentaje de los textos aparece cada una, con falacia y sin falacia. **Solo con el `train`** de las prototípicas y las espontáneas: son los corpus con textos sin falacia, y mirar el test sería hacer trampa.

In [12]:
from nltk.corpus import stopwords

STOPWORDS_NLTK = stopwords.words("spanish")

train = pd.concat([corpus["fallacyes_proto"], corpus["fallacyes_espont"]])
train = train[train["split"] == "train"].copy()
train["clase"] = train["etiquetas"].map(lambda etiquetas: etiquetas[0] if etiquetas else "sin_falacia")
train["palabras"] = train["texto"].map(lambda texto: {token.lower() for token in casual_tokenize(texto)})


def porcentaje_de_textos_con(palabra, textos):
    return 100 * textos["palabras"].map(lambda palabras: palabra in palabras).mean()


# Candidatas: las 7 acordadas al principio y las que más destacan al medir todas las de NLTK
CANDIDATAS = ["o", "no", "porque", "todo", "todos", "ni", "nada",
              "tanto", "eres", "tú", "te", "tu", "qué", "nos", "nosotros", "contra", "tienes", "?", "!"]
CLASES = ["ad_hominem", "hombre_de_paja", "falso_dilema", "falsa_causalidad", "deduccion_invalida",
          "ad_populum", "generalizacion_apresurada", "distraccion"]

sin_falacia = train[train["clase"] == "sin_falacia"]
tabla = pd.DataFrame(index=CANDIDATAS)
tabla["% sin falacia"] = [porcentaje_de_textos_con(palabra, sin_falacia) for palabra in CANDIDATAS]
tabla["% con falacia"] = [porcentaje_de_textos_con(palabra, train[train["clase"] != "sin_falacia"]) for palabra in CANDIDATAS]
for clase in CLASES:
    # Cuántas veces más aparece en esa clase que en los textos sin falacia
    textos_de_la_clase = train[train["clase"] == clase]
    tabla[clase] = [(porcentaje_de_textos_con(palabra, textos_de_la_clase) + 0.5) / (porcentaje + 0.5)
                    for palabra, porcentaje in zip(CANDIDATAS, tabla["% sin falacia"])]
tabla.round(1)

,% sin falacia,% con falacia,ad_hominem,hombre_de_paja,falso_dilema,falsa_causalidad,deduccion_invalida,ad_populum,generalizacion_apresurada,distraccion
o,13.1,10.1,0.5,0.9,3.5,0.2,0.8,0.5,0.3,0.3
no,57.4,38.0,0.9,0.9,0.6,0.3,0.7,0.4,0.6,0.8
porque,7.7,11.9,1.3,1.4,0.5,1.6,0.9,2.0,0.9,0.4
todo,3.9,7.9,1.6,2.9,0.8,0.3,1.3,6.9,1.4,1.3
todos,7.5,8.6,0.5,0.9,0.8,0.6,1.6,2.2,2.2,0.9
ni,2.0,2.8,3.1,0.7,1.0,0.8,0.5,0.7,1.4,1.5
nada,2.1,2.4,1.3,1.9,1.3,0.2,0.8,0.6,0.7,1.4
tanto,2.2,7.8,0.9,1.4,0.5,6.7,13.4,1.5,4.8,1.7
eres,0.4,2.3,9.4,2.9,4.6,0.5,2.2,1.8,0.9,0.5
tú,1.0,2.8,4.4,1.8,1.0,0.8,0.3,6.9,0.5,3.8


Las columnas de cada clase dicen cuántas veces más sale la palabra en esa clase que en los textos sin falacia: 1 = igual; más de 1 = más en la falacia; menos de 1 = más en los textos sin falacia. Se suma 0,5 arriba y abajo para no dividir por cero.

**Decisión:** se conservan 17 palabras de la lista de NLTK, más los signos `?` y `!`:

| Grupo | Palabras | Por qué |
|---|---|---|
| Acordadas al principio | *o, no, porque, todo, todos, ni, nada* | Disyunción (falso dilema), negación, causa y cuantificadores universales. *No* y *o* salen menos en las falacias (muchas parejas no falaces niegan: *"esto **no** significa…"*), pero eso también sirve para distinguir, y *o* se dispara en el falso dilema |
| Conclusión | *tanto* | *"por lo **tanto**"*: muy frecuente en la deducción inválida y la falsa causalidad |
| Segunda persona | *eres, tú, te, tu, tienes* | Ataque directo a la persona (ad hominem): *"**tú** **eres** un…"* |
| Preguntas retóricas | *qué* | Hombre de paja y ad hominem: *"¿**qué** quiere, que…?"* |
| Bandos | *nos, nosotros, contra* | Falso dilema y hombre de paja: *"o con **nosotros** o **contra**…"* |
| Signos | `?` `!` | `?` sale en el 4 % de los textos sin falacia y en una cuarta parte de los hombres de paja y las distracciones |

Se descartan otras palabras que también destacaban (*mi*, *esa*, *he*…) pero sin una explicación clara: podrían ser casualidad de estos textos. *Si*, *nunca*, *siempre*, *nadie* y *usted* no están en la lista de NLTK, así que se conservan sin hacer nada. Los signos de apertura ¿ ¡ se quitan: repiten la información de `?` y `!`.

Ojo: la medición no incluye Political (no tiene textos sin falacia), y parte de las diferencias pueden venir de cómo se escribieron a mano las parejas no falaces.

In [13]:
from nltk.stem.snowball import SnowballStemmer

PALABRAS_QUE_MARCAN_FALACIAS = {
    "o", "no", "porque", "todo", "todos", "ni", "nada",  # acordadas al principio
    "tanto", "eres", "tú", "te", "tu", "tienes", "qué", "nos", "nosotros", "contra",  # añadidas tras medirlo (5.1)
}
STOPWORDS = set(STOPWORDS_NLTK) - PALABRAS_QUE_MARCAN_FALACIAS
SIGNOS_QUE_SE_CONSERVAN = {"?", "!"}
MARCA_DE_CIFRA = "<num>"

stemmer = SnowballStemmer("spanish")


def es_palabra_o_numero(token):
    return any(caracter.isalnum() for caracter in token)


def es_cifra(token):
    """Tiene dígitos y ninguna letra: "2023", "0,5", "2.900". "G20" no es una cifra."""
    return any(caracter.isdigit() for caracter in token) and not any(caracter.isalpha() for caracter in token)


def raices_para_bow(tokens):
    """Minúsculas, sin stopwords ni puntuación (salvo ? y !), cifras cambiadas por <num>
    y cada palabra reducida a su raíz."""
    raices = []
    for token in tokens:
        token = token.lower()
        if token in SIGNOS_QUE_SE_CONSERVAN:
            raices.append(token)
        elif es_cifra(token):
            raices.append(MARCA_DE_CIFRA)
        elif es_palabra_o_numero(token) and token not in STOPWORDS:
            raices.append(stemmer.stem(token))
    return raices


print("Stopwords que se quitan:", len(STOPWORDS), "de", len(STOPWORDS_NLTK))
for oracion in ["¿Y tú eres el que habla de corrupción? ¡Por favor!",
                "O estás con nosotros o estás contra nosotros.",
                "Todos los países lo hacen, por lo tanto no puede estar mal.",
                "El paro ha bajado un 0,5 % desde 2023 en España."]:
    print(oracion, "\n   →", raices_para_bow(casual_tokenize(oracion)))

Stopwords que se quitan: 296 de 313
¿Y tú eres el que habla de corrupción? ¡Por favor! 
   → ['tu', 'eres', 'habl', 'corrupcion', '?', 'favor', '!']
O estás con nosotros o estás contra nosotros. 
   → ['o', 'nosotr', 'o', 'contr', 'nosotr']
Todos los países lo hacen, por lo tanto no puede estar mal. 
   → ['tod', 'pais', 'hac', 'tant', 'no', 'pued', 'mal']
El paro ha bajado un 0,5 % desde 2023 en España. 
   → ['par', 'baj', '0,5', '2023', 'españ']


Dos efectos del stemmer que hay que conocer:
- **Quita las tildes**: *país* → `pais`, *está* → `esta`. Así se juntan las variantes mal escritas del corpus espontáneo (*pais*, *tambien*).
- **A veces junta palabras distintas**: *señorías* y *señor* dan las dos `señor`.

### 5.2 Cifras: la marca `<num>`

Todo token que tiene dígitos y ninguna letra se cambia por `<num>`. Para detectar falacias no importa la cifra concreta, sino que **hay** una cifra: los datos y las estadísticas son típicos de la falacia de credibilidad y de la generalización. Además, cada cifra distinta sería una "palabra" que sale una sola vez y solo engorda el vocabulario.

También evita un fallo de `casual_tokenize`, que solo entiende un separador dentro de un número: *2.900.000* sale como `2.900` `.` `000` (13 casos en proto y Political), y las cifras con espacio del Diario, *14 400*, como `14` `400` (1.116 en el Congreso). Con la marca da igual cómo las parta: quedan como `<num>`.

Los `tokens` para embeddings se dejan como salen de `casual_tokenize`: para los embeddings las cifras apenas importan y casi nunca están en su vocabulario.

## 6. La función de preprocesado

`preprocesar` junta todas las piezas anteriores. Recibe el `texto` limpio de un ejemplo (o de una intervención) y su `id`, y devuelve la lista `oraciones` del esquema:

1. NFC (sección 1).
2. División en oraciones con las abreviaturas añadidas (sección 3).
3. Para cada oración:
   - `tokens` con `casual_tokenize` (sección 4);
   - `raices_bow` (sección 5);
   - un `id`: el del texto + `_s` + el número de la oración.

**El número de oración lleva 3 cifras** (`_s000`, `_s001`…), y no 2 como decía al principio el esquema: hay intervenciones del Congreso con más de 100 oraciones (la más larga tiene 387). Con 3 cifras todos los IDs tienen la misma longitud y se ordenan bien como texto.

In [14]:
def preprocesar(texto, id_texto):
    """Divide el texto en oraciones y prepara cada una para embeddings (tokens) y para BoW (raices_bow)."""
    texto = unicodedata.normalize("NFC", texto)
    oraciones = []
    for numero, oracion in enumerate(dividir_en_oraciones(texto)):
        tokens = casual_tokenize(oracion)
        oraciones.append({
            "id": f"{id_texto}_s{numero:03d}",
            "texto": oracion,
            "tokens": tokens,
            "raices_bow": raices_para_bow(tokens),
        })
    return oraciones


# Un ejemplo real del corpus espontáneo
ejemplo = corpus["fallacyes_espont"].iloc[0]
print(ejemplo["texto"], "\n")
for oracion in preprocesar(ejemplo["texto"], ejemplo["id"]):
    print(json.dumps(oracion, ensure_ascii=False))

Para mi lo que diga este tipejo tiene cero validez. 

{"id": "fes_0000_s000", "texto": "Para mi lo que diga este tipejo tiene cero validez.", "tokens": ["Para", "mi", "lo", "que", "diga", "este", "tipejo", "tiene", "cero", "validez", "."], "raices_bow": ["dig", "tipej", "cer", "validez"]}


### 6.1 Pruebas con frases difíciles

In [15]:
FRASES_DIFICILES = {
    "siglas": "El PSOE y el PP votaron con EE. UU. y la OTAN en contra.",
    "números": "El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.",
    "abreviatura": "La Sra. García lo dijo en el art. 155.",
    "cita con «»": "Me dijo: «No pienso votarte. Eres un traidor». Y se fue.",
    "cita con \"\"": 'Como dijo Einstein: "La imaginación es más importante que el conocimiento". Y tenía razón.',
    "preguntas y exclamaciones": "¿Usted cree que esto es serio? ¡Por favor! Que dimita.",
    "puntos suspensivos": "Lo dijo el presidente… Y nadie le creyó.",
    "sin puntuación (tipo Whisper)": "o gobernamos nosotros o gobierna la derecha y eso lo sabe todo el mundo",
    "vacío": "",
}
for caso, frase in FRASES_DIFICILES.items():
    print(f"--- {caso}: {frase!r}")
    for oracion in preprocesar(frase, "prueba"):
        print(f"   {oracion['id']} {oracion['texto']!r}")
        print(f"        tokens:     {oracion['tokens']}")
        print(f"        raices_bow: {oracion['raices_bow']}")

--- siglas: 'El PSOE y el PP votaron con EE. UU. y la OTAN en contra.'
   prueba_s000 'El PSOE y el PP votaron con EE. UU. y la OTAN en contra.'
        tokens:     ['El', 'PSOE', 'y', 'el', 'PP', 'votaron', 'con', 'EE', '.', 'UU', '.', 'y', 'la', 'OTAN', 'en', 'contra', '.']
        raices_bow: ['pso', 'pp', 'vot', 'ee', 'uu', 'otan', 'contr']
--- números: 'El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.'
   prueba_s000 'El paro bajó un 0,5 % en 2023: de 2.900.000 a 2.700.000 parados.'
        tokens:     ['El', 'paro', 'bajó', 'un', '0,5', '%', 'en', '2023', ':', 'de', '2.900', '.', '000', 'a', '2.700', '.', '000', 'parados', '.']
        raices_bow: ['par', 'baj', '0,5', '2023', '2.900', '000', '2.700', '000', 'par']
--- abreviatura: 'La Sra. García lo dijo en el art. 155.'
   prueba_s000 'La Sra. García lo dijo en el art. 155.'
        tokens:     ['La', 'Sra', '.', 'García', 'lo', 'dijo', 'en', 'el', 'art', '.', '155', '.']
        raices_bow: ['sra', 'garc', 'dij

In [16]:
# Un texto real de Political con poca puntuación (más de 60 palabras y un solo signo de final)
political = corpus["fallacyes_political"]
poco_puntuado = political[(political["texto"].str.split().str.len() > 60) & (political["texto"].str.count(r"[.?!]") <= 1)]
print("Textos así en Political:", len(poco_puntuado), "de", len(political))
ejemplo = poco_puntuado.iloc[0]
for oracion in preprocesar(ejemplo["texto"], ejemplo["id"]):
    print(oracion["id"], len(oracion["tokens"]), "tokens:", oracion["texto"][:300], "...")

Textos así en Political: 21 de 1951
fpo_0009_s000 87 tokens: Es que usted no tiene la confianza, en este momento, ni de la mayoría de los ciudadanos españoles en su política económica, ni de los principales inversores extranjeros, ni de los organismos internacionales, ni del Comité Monetario de la Comunidad Europea, ni de los sindicatos, ni de los empresarios ...


**Lo que sale de las pruebas:**
- Siglas, abreviaturas, preguntas y exclamaciones: bien.
- **Cifras**: en `tokens`, *2.900.000* se parte (`2.900` `.` `000`), pero en `raices_bow` todo queda como `<num>` (sección 5.2).
- **Citas con un punto dentro**: *«No pienso votarte. Eres un traidor».* se parte en dos oraciones, y cada comilla queda en una. Es correcto (son dos oraciones); solo queda raro a la vista.
- **Puntos suspensivos `…` seguidos de mayúscula**: Punkt no corta (*"Lo dijo el presidente… Y nadie le creyó."* queda en una). En el Congreso, `…` casi siempre marca una interrupción en mitad de la frase, así que no cortar es lo más seguro.
- **Texto sin puntuación**: queda como una sola oración larga. En Political solo hay 21 textos así, y la mayoría llevan comas: son frases largas, no texto sin puntuar. Se acepta (ver sección 2.1).
- **Texto vacío**: devuelve una lista vacía, sin error.

## 7. Aplicar a los cuatro corpus y exportar los `*_procesado.jsonl`

Cada `*_procesado.jsonl` tiene lo mismo que su `*_limpio.jsonl` más el campo `oraciones`. Se genera siempre a partir del limpio y nunca se edita a mano: si cambia algo del preprocesado, se vuelve a ejecutar este notebook.

In [17]:
from tqdm import tqdm

procesados = {}
for nombre, tabla in corpus.items():
    procesado = tabla.copy()
    procesado["oraciones"] = [
        preprocesar(texto, id_texto)
        for texto, id_texto in tqdm(zip(tabla["texto"], tabla["id"]), total=len(tabla), desc=nombre)
    ]
    procesados[nombre] = procesado

congreso: 100%|██████████| 2468/2468 [00:21<00:00, 117.41it/s]


In [18]:
def guardar_jsonl(tabla, nombre_fichero):
    """Guarda la tabla en JSONL (una línea por fila). Los valores vacíos (NaN) se guardan como null."""
    ruta = CARPETA_DATOS / nombre_fichero
    registros = tabla.astype(object).where(tabla.notna(), None).to_dict(orient="records")
    with open(ruta, "w", encoding="utf-8") as fichero:
        for registro in registros:
            fichero.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"Guardado {ruta.name}: {len(registros)} textos, {ruta.stat().st_size / 1e6:.0f} MB")


for nombre, procesado in procesados.items():
    guardar_jsonl(procesado, f"{nombre}_procesado.jsonl")

Guardado fallacyes_proto_procesado.jsonl: 3319 textos, 4 MB
Guardado fallacyes_espont_procesado.jsonl: 1840 textos, 3 MB
Guardado fallacyes_political_procesado.jsonl: 1951 textos, 5 MB
Guardado congreso_procesado.jsonl: 2468 textos, 60 MB


### Comprobación: se vuelven a leer con pandas

Se leen como lo hará el resto del grupo y se comprueba que no se ha perdido ni cambiado nada.

In [19]:
for nombre, procesado in procesados.items():
    leido = pd.read_json(CARPETA_DATOS / f"{nombre}_procesado.jsonl", lines=True, dtype=False)
    oraciones = leido["oraciones"].explode().dropna()
    ids_de_oracion = oraciones.map(lambda oracion: oracion["id"])
    print(nombre)
    print("   textos:", len(leido), "| iguales que al guardar:", len(leido) == len(procesado))
    print("   columnas: las del limpio + oraciones:", list(leido.columns) == list(corpus[nombre].columns) + ["oraciones"])
    print("   textos sin oraciones:", (leido["oraciones"].str.len() == 0).sum())
    print("   IDs de oración repetidos:", ids_de_oracion.duplicated().sum())
    print("   oraciones sin tokens:", (oraciones.map(lambda oracion: len(oracion["tokens"])) == 0).sum())
    print("   oraciones:", len(oraciones), "| tokens:", oraciones.map(lambda oracion: len(oracion["tokens"])).sum())

fallacyes_proto
   textos: 3319 | iguales que al guardar: True
   columnas: las del limpio + oraciones: True
   textos sin oraciones: 0
   IDs de oración repetidos: 0
   oraciones sin tokens: 0
   oraciones: 5819 | tokens: 96434
fallacyes_espont
   textos: 1840 | iguales que al guardar: True
   columnas: las del limpio + oraciones: True
   textos sin oraciones: 0
   IDs de oración repetidos: 0
   oraciones sin tokens: 0
   oraciones: 2745 | tokens: 54243
fallacyes_political
   textos: 1951 | iguales que al guardar: True
   columnas: las del limpio + oraciones: True
   textos sin oraciones: 0
   IDs de oración repetidos: 0
   oraciones sin tokens: 0
   oraciones: 4289 | tokens: 91241
congreso
   textos: 2468 | iguales que al guardar: True
   columnas: las del limpio + oraciones: True
   textos sin oraciones: 0
   IDs de oración repetidos: 0
   oraciones sin tokens: 0
   oraciones: 68441 | tokens: 1864193


### Cómo leerlos por oraciones

Para trabajar con una fila por oración (por ejemplo, para entrenar el Word2Vec), se "despliega" la lista `oraciones` y se le añaden los campos del texto que hagan falta:

In [20]:
congreso_procesado = pd.read_json(CARPETA_DATOS / "congreso_procesado.jsonl", lines=True, dtype=False)

# Una fila por oración, con los campos del texto que se quieran conservar
por_oracion = congreso_procesado[["id", "uso", "orador", "oraciones"]].explode("oraciones", ignore_index=True)
# Cada oración es un diccionario: se pasa a columnas (id, texto, tokens, raices_bow)
campos_de_la_oracion = pd.json_normalize(por_oracion["oraciones"].tolist())
por_oracion = por_oracion.drop(columns="oraciones").rename(columns={"id": "id_texto"}).join(campos_de_la_oracion)
por_oracion.head()

,id_texto,uso,orador,id,texto,tokens,raices_bow
0,cg_L15_10_i002,general,El señor LOSADA FERNÁNDEZ,cg_L15_10_i002_s000,"Sí, prometo.","[Sí, ,, prometo, .]",[promet]
1,cg_L15_10_i004,general,El señor LORENZO CAZORLA,cg_L15_10_i004_s000,"Sí, prometo.","[Sí, ,, prometo, .]",[promet]
2,cg_L15_10_i007,general,La señora MONTESINOS DE MIGUEL,cg_L15_10_i007_s000,"Gracias, señora presidenta.","[Gracias, ,, señora, presidenta, .]","[graci, señor, president]"
3,cg_L15_10_i007,general,La señora MONTESINOS DE MIGUEL,cg_L15_10_i007_s001,"Señorías, este debate no es nuevo.","[Señorías, ,, este, debate, no, es, nuevo, .]","[señor, debat, no, nuev]"
4,cg_L15_10_i007,general,La señora MONTESINOS DE MIGUEL,cg_L15_10_i007_s002,"Es costumbre parlamentaria, al inicio de cada ...","[Es, costumbre, parlamentaria, ,, al, inicio, ...","[costumbr, parlamentari, inici, cad, legislatu..."
